# MGMT298D: Science and Strategy of AI
## Assignment 6 - CNN Architecture and Transfer Learning
### Application: Product Image Classification

---

**Instructions:** Complete the exercises by filling in the `???` placeholders and answering the questions. Run all code cells in order.

## Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

# Use subset for faster training
x_train_sub = x_train[:20000]
y_train_sub = y_train[:20000]

print(f"Training subset: {x_train_sub.shape} | Test: {x_test.shape}")

## Part 1: Effect of Filter Size

Convolutional filters can be different sizes (e.g., 3x3, 5x5, 7x7). Let's see how filter size affects performance.

In [ ]:
def build_cnn(filter_size):
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, filter_size, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, filter_size, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

filter_sizes = [3, 5, 7]
filter_results = {"filter_size": [], "test_accuracy": [], "params": []}

for fs in filter_sizes:
    print(f"Training with {fs}x{fs} filters...")
    model = build_cnn(fs)
    model.fit(x_train_sub, y_train_sub, epochs=10, batch_size=128, 
              validation_split=0.15, verbose=0)
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    filter_results["filter_size"].append(f"{fs}x{fs}")
    filter_results["test_accuracy"].append(test_acc)
    filter_results["params"].append(model.count_params())

filter_df = pd.DataFrame(filter_results)
print("\n=== Filter Size Results ===")
print(filter_df.to_string(index=False))

## Part 2: Effect of Number of Filters

More filters = more patterns the model can learn, but also more parameters.

In [ ]:
def build_cnn_filters(num_filters):
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(num_filters, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(num_filters * 2, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

num_filters_list = [8, 16, 32, 64]
num_filter_results = {"num_filters": [], "test_accuracy": [], "params": []}

for nf in num_filters_list:
    print(f"Training with {nf} filters in first layer...")
    model = build_cnn_filters(nf)
    model.fit(x_train_sub, y_train_sub, epochs=10, batch_size=128,
              validation_split=0.15, verbose=0)
    
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    num_filter_results["num_filters"].append(nf)
    num_filter_results["test_accuracy"].append(test_acc)
    num_filter_results["params"].append(model.count_params())

num_filter_df = pd.DataFrame(num_filter_results)
print("\n=== Number of Filters Results ===")
print(num_filter_df.to_string(index=False))

In [ ]:
# Visualize accuracy vs parameters trade-off
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(num_filter_df["params"], num_filter_df["test_accuracy"], s=100, c="#45b7d1")
for i, row in num_filter_df.iterrows():
    ax.annotate(f"{row['num_filters']} filters", 
                (row['params'], row['test_accuracy']),
                textcoords="offset points", xytext=(5, 5))

ax.set_xlabel("Number of Parameters")
ax.set_ylabel("Test Accuracy")
ax.set_title("Accuracy vs Model Complexity")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Transfer Learning with MobileNetV2

Instead of training from scratch, we can use a pre-trained model and fine-tune it.

In [ ]:
from tensorflow.keras.applications import MobileNetV2

# Resize images for MobileNetV2 (expects 96x96 minimum)
x_train_resized = tf.image.resize(x_train_sub, (96, 96))
x_test_resized = tf.image.resize(x_test, (96, 96))

# Load pre-trained MobileNetV2 (without top layers)
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(96, 96, 3))
base_model.trainable = False  # Freeze the base model

# Add custom classification head
transfer_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

transfer_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Transfer model parameters: {transfer_model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in transfer_model.trainable_weights]):,}")

In [ ]:
# Train transfer learning model
print("Training transfer learning model...")
history_transfer = transfer_model.fit(
    x_train_resized, y_train_sub, epochs=10, batch_size=64,
    validation_split=0.15, verbose=1
)

transfer_loss, transfer_acc = transfer_model.evaluate(x_test_resized, y_test)
print(f"\nTransfer Learning — Test Accuracy: {transfer_acc*100:.2f}%")

## Part 4: Model Comparison

In [ ]:
# Train a from-scratch CNN for comparison
scratch_model = build_cnn_filters(32)
scratch_model.fit(x_train_sub, y_train_sub, epochs=10, batch_size=128,
                  validation_split=0.15, verbose=0)
scratch_loss, scratch_acc = scratch_model.evaluate(x_test, y_test, verbose=0)

comparison = pd.DataFrame({
    "Model": ["CNN from Scratch", "Transfer Learning (MobileNetV2)"],
    "Test Accuracy": [scratch_acc * 100, transfer_acc * 100],
    "Trainable Params": [scratch_model.count_params(), 
                         sum([tf.size(w).numpy() for w in transfer_model.trainable_weights])]
})
print(comparison.to_string(index=False))

In [ ]:
# Confusion matrix for transfer model
y_pred_transfer = np.argmax(transfer_model.predict(x_test_resized), axis=1)
cm = confusion_matrix(y_test, y_pred_transfer)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Transfer Learning Model - Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## Questions

### Question 1: Filter Size Trade-offs

**Q1:** Larger filters (5x5, 7x7) can capture larger patterns but have more parameters. Based on your results, which filter size worked best for CIFAR-10? Why might 3x3 filters be the standard choice in modern architectures like VGG and ResNet?

*Your answer:*



---
### Question 2: Number of Filters

**Q2a:** How did increasing the number of filters affect accuracy and the number of parameters? At what point did you see diminishing returns?

*Your answer:*



**Q2b:** A startup wants to deploy an image classifier on mobile phones with limited memory. Based on your results, how would you balance accuracy vs. model size?

*Your answer:*



---
### Question 3: Transfer Learning

**Q3a:** Compare the CNN trained from scratch vs. transfer learning. Which performed better? Why does transfer learning work even though MobileNetV2 was trained on ImageNet (1000 classes, different from CIFAR-10)?

*Your answer:*



**Q3b:** We froze the base model and only trained the classification head. When might you want to "fine-tune" by unfreezing some layers of the base model?

*Your answer:*



---
### Question 4: Business Decision - Build vs Buy

**Q4:** Your company needs an image classification system. You could: (A) Train a CNN from scratch on your data, (B) Use transfer learning with a pre-trained model, or (C) Use a cloud API like Google Vision or AWS Rekognition. What factors would influence your decision? When would each option be best?

*Your answer:*



---
### Question 5: Error Analysis and Deployment

**Q5:** Look at the confusion matrix. Which classes are most confused? If you were deploying this model for a self-driving car application (which includes some of these classes), what additional steps would you take before deployment?

*Your answer:*

